# Análisis de Ensambles Neuronales — V1
Carga datos usando el mismo pipeline de calcio y construye matriz **Trials × Neuronas** para UMAP/PCA/GMM.

**Celdas 1–4:** carga idéntica al pipeline principal (solo edita rutas en celda 2).  
**Celdas 5+:** análisis de ensambles.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.mixture import GaussianMixture
from scipy import stats, optimize
from pathlib import Path
import warnings, traceback
warnings.filterwarnings('ignore')
print('Dependencias cargadas.')

Dependencias cargadas.


In [2]:
# ─── RUTAS ────────────────────────────────────────────────────────────────────
DATA_ROOT = r'D:\AL-205\Nicole\Calcio\Nueva carpeta'

SCOPE_KEYWORD  = 'reg'
STIM_KEYWORD   = 'pas'
TRACES_KEYWORD = 'traces'

GENOTYPE_FOLDERS = {'BTBR': 'BTBR', 'C57': 'C57'}

# ─── ADQUISICIÓN ──────────────────────────────────────────────────────────────
SAMPLING_RATE = 5           # Hz
MS_PER_FRAME  = 1000 / SAMPLING_RATE  # 200 ms

# ─── VENTANAS ─────────────────────────────────────────────────────────────────
SEC_BEFORE    = 2
SEC_AFTER     = 4
FRAMES_BEFORE = int(SEC_BEFORE * SAMPLING_RATE)   # 10
FRAMES_AFTER  = int(SEC_AFTER  * SAMPLING_RATE)   # 20
TOTAL_FRAMES  = FRAMES_BEFORE + FRAMES_AFTER       # 30
RESP_START    = FRAMES_BEFORE   # frame donde empieza el estímulo
RESP_END      = TOTAL_FRAMES    # frame donde termina

# ─── NORMALIZACIÓN ────────────────────────────────────────────────────────────
# 'dff'   → dF/F0  (F0 = mediana del baseline por trial)
# 'zscore'→ z-score  (media y std del baseline por trial)
NORMALIZATION = 'dff'

# ─── UMBRALES ─────────────────────────────────────────────────────────────────
RESPONSIVE_STD  = 2.5   # STDs sobre baseline (por trial) para neurona responsiva
OSI_THRESH      = 0.4   # OSI mínimo (vectorial) para neurona selectiva
R2_THRESH       = 0.7   # R² mínimo para ajuste gaussiano bimodal (Ortiz-Cruz)

# ─── OUTPUT ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path(DATA_ROOT) / 'pipeline_output'
OUTPUT_DIR.mkdir(exist_ok=True)

PALETTE = {'BTBR': '#e74c3c', 'C57': '#3498db', 'Unknown': '#95a5a6'}

print('Configuracion cargada.')
print(f'Normalización: {NORMALIZATION}')
print(f'Output → {OUTPUT_DIR}')

Configuracion cargada.
Normalización: dff
Output → D:\AL-205\Nicole\Calcio\Nueva carpeta\pipeline_output


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
#  MÓDULO 1 — Detección de archivos
# ══════════════════════════════════════════════════════════════════════════════

def find_file(folder: Path, keyword: str) -> Path:
    kw = keyword.lower()
    matches = [f for f in folder.iterdir()
               if f.is_file() and kw in f.name.lower()]
    if not matches:
        raise FileNotFoundError(f"No encontré archivo con '{keyword}' en {folder}")
    txt = [f for f in matches if f.suffix.lower() == '.txt']
    csv = [f for f in matches if f.suffix.lower() == '.csv']
    if kw in ['reg', 'pas']:
        return txt[0] if txt else matches[0]
    return csv[0] if csv else matches[0]


def detect_genotype(group_name: str) -> str:
    for kw, geno in GENOTYPE_FOLDERS.items():
        if kw.upper() in group_name.upper():
            return geno
    return 'Unknown'


def discover_animals(root: str) -> list:
    root = Path(root)
    animals = []
    for group in sorted(root.iterdir()):
        if not group.is_dir() or group.name.startswith('.') \
                or group.name == 'pipeline_output':
            continue
        genotype = detect_genotype(group.name)
        for animal in sorted(group.iterdir()):
            if not animal.is_dir() or animal.name.startswith('.'):
                continue
            animals.append({'animal_id': animal.name,
                            'genotype':  genotype,
                            'group':     group.name,
                            'folder':    animal})
    return animals


# ══════════════════════════════════════════════════════════════════════════════
#  MÓDULO 2 — Parsers de TXT
# ══════════════════════════════════════════════════════════════════════════════

def _hms_to_ms(hms_str: str) -> float:
    """Convierte 'HH:MM:SS' a milisegundos absolutos desde medianoche."""
    h, m, s = map(int, hms_str.strip().split(':'))
    return (h * 3600 + m * 60 + s) * 1000.0


def parse_scope_txt(filepath: Path) -> tuple:
    """
    Formato: HH:MM:SS.microsec.framecount.flag.canal
    Ejemplo: 12:17:32.228166.16884.23.1
      campo 0 = HH:MM:SS
      campo 1 = microsec (6 digitos, fraccion del segundo)
      campo 2 = framecount
      campo 3 = flag  (23=frame, 4=evento externo)
      campo 4 = canal

    Retorna (timestamp_strs, timestamp_ms_abs, millisec_rel, sensores).
    """
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    timestamp_strs, timestamp_ms_abs, millisec_list, sensores_list = [], [], [], []

    for line in lines:
        parts = line.strip().split('.')
        if len(parts) < 5:
            continue
        hms_str  = parts[0]
        microsec = float(parts[1]) / 1000.0   # microsec → ms
        flag     = float(parts[3])

        t_abs = _hms_to_ms(hms_str) + microsec
        timestamp_strs.append(hms_str)
        timestamp_ms_abs.append(t_abs)
        millisec_list.append(t_abs)
        sensores_list.append(flag)

    timestamp_ms_abs = np.array(timestamp_ms_abs)
    millisec         = timestamp_ms_abs - timestamp_ms_abs[0]   # relativo a t=0
    sensores         = np.array(sensores_list)

    return timestamp_strs, timestamp_ms_abs, millisec, sensores


def parse_stim_txt(filepath: Path) -> tuple:
    """
    Formato: HH:MM:SS.mmm.FLAG
    Ejemplo:
      12:17:32.098.11    → onset  (flag == 11)
      12:17:36.260.0180  → fin / angulo 180 (flag con prefijo '0')

    El angulo real: str(FLAG).lstrip('0') o '0' si era todo ceros.
    Retorna (tstamp_stlog, tstamp_ms_abs, estimulos_real).
    estimulos_real: 11 para onset, angulo real (0,45,90,...) para fin.
    """
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    tstamp_stlog, tstamp_ms_list, estimulos_list = [], [], []

    for line in lines:
        parts = line.strip().split('.')
        if len(parts) < 3:
            continue
        hms_str = parts[0]
        mmm     = float(parts[1])          # milisegundos
        flag_raw = parts[2].strip()

        t_abs = _hms_to_ms(hms_str) + mmm

        # Decodificar flag
        if flag_raw == '11':
            flag_val = 11
        else:
            # Formato "0XYZ" donde XYZ es el angulo
            # "0180" -> 180, "090" -> 90, "00" -> 0, "045" -> 45
            angle_str = flag_raw.lstrip('0')
            flag_val  = int(angle_str) if angle_str else 0

        tstamp_stlog.append(hms_str)
        tstamp_ms_list.append(t_abs)
        estimulos_list.append(float(flag_val))

    return tstamp_stlog, np.array(tstamp_ms_list), np.array(estimulos_list)


def get_clean_frame_times(millisec: np.ndarray, sensores: np.ndarray,
                           min_interval_ms: float = 100.0) -> np.ndarray:
    """Frame times relativos (ms desde inicio). Flag de frame = 23."""
    framt    = np.where(sensores == 23)[0]
    t_fram   = millisec[framt]
    ll       = np.diff(t_fram)
    idx_keep = np.where(ll > min_interval_ms)[0] + 1
    return np.insert(t_fram[idx_keep], 0, t_fram[0])


def get_clean_frame_times_abs(timestamp_ms: np.ndarray,
                               sensores: np.ndarray,
                               min_interval_ms: float = 100.0) -> np.ndarray:
    """Frame times absolutos (ms desde medianoche). Flag de frame = 23."""
    framt    = np.where(sensores == 23)[0]
    t_fram   = timestamp_ms[framt]
    ll       = np.diff(t_fram)
    idx_keep = np.where(ll > min_interval_ms)[0] + 1
    return np.insert(t_fram[idx_keep], 0, t_fram[0])


# ══════════════════════════════════════════════════════════════════════════════
#  MÓDULO 3 — Alineación estímulos <-> frames  (tiempo absoluto)
# ══════════════════════════════════════════════════════════════════════════════

def align_stimuli_to_frames(timestamp_ms_scope: np.ndarray,
                             millisec: np.ndarray,
                             sensores: np.ndarray,
                             tstamp_ms_stim: np.ndarray,
                             estimulos: np.ndarray,
                             ms_per_frame: float = MS_PER_FRAME) -> tuple:
    """
    Alineación robusta usando tiempo absoluto en ms.

    Estrategia:
      Para cada evento '11' (onset), encuentra el frame del microscopio
      mas cercano en tiempo absoluto. El angulo del trial es el evento
      inmediatamente siguiente (o anterior) al 11.

    Devuelve (stimsnframes, n_unmatched).
    stimsnframes: (n_angulos, n_trials, 2)  [:,:,0]=angulo  [:,:,1]=frame
    """
    from collections import defaultdict

    t_frames_abs = get_clean_frame_times_abs(timestamp_ms_scope, sensores)
    max_frame    = len(t_frames_abs) - 1

    idx_11 = np.where(estimulos == 11)[0]
    if len(idx_11) == 0:
        raise ValueError("No se encontraron eventos '11' (onset) en el archivo de estímulos.")

    trial_records = []  # lista de (angulo, frame_idx)
    n_unmatched   = 0

    for idx in idx_11:
        t_onset = tstamp_ms_stim[idx]

        # Frame mas cercano
        diffs   = np.abs(t_frames_abs - t_onset)
        nearest = int(np.argmin(diffs))
        if diffs[nearest] > 500:   # umbral 500ms
            n_unmatched += 1
            continue

        # Buscar angulo: primero en los eventos siguientes al 11
        angle = None
        for k in range(idx + 1, min(idx + 5, len(estimulos))):
            if estimulos[k] != 11:
                angle = int(estimulos[k])
                break
        # Si no hay angulo después, buscar antes
        if angle is None:
            for k in range(idx - 1, max(idx - 5, -1), -1):
                if estimulos[k] != 11:
                    angle = int(estimulos[k])
                    break
        if angle is None:
            n_unmatched += 1
            continue

        trial_records.append((angle, nearest))

    if not trial_records:
        raise ValueError(
            "No se encontraron trials alineables (ningún onset dentro de 500ms de un frame). "
            "Corre la celda de diagnóstico para verificar la superposición de timestamps."
        )

    trials_by_angle = defaultdict(list)
    for angle, frame_idx in trial_records:
        trials_by_angle[angle].append(frame_idx)

    angles_sorted = sorted(trials_by_angle.keys())
    n_trials      = min(len(trials_by_angle[a]) for a in angles_sorted)
    n_ang         = len(angles_sorted)

    stimsnframes = np.zeros((n_ang, n_trials, 2), dtype=int)
    for ai, angle in enumerate(angles_sorted):
        stimsnframes[ai, :, 0] = angle
        stimsnframes[ai, :, 1] = trials_by_angle[angle][:n_trials]

    return stimsnframes, n_unmatched


# ══════════════════════════════════════════════════════════════════════════════
#  MÓDULO 4 — Extracción y normalización
# ══════════════════════════════════════════════════════════════════════════════

def normalize_trial(F: np.ndarray, frames_before: int,
                    method: str = 'dff') -> np.ndarray:
    """Normaliza F (T x N_neuronas) contra su baseline."""
    baseline = F[:frames_before]
    if method == 'zscore':
        mu    = np.mean(baseline, axis=0)
        sigma = np.std(baseline,  axis=0)
        return (F - mu) / (sigma + 1e-6)
    else:  # dff
        F0 = np.median(baseline, axis=0)
        return (F - F0) / (np.abs(F0) + 1e-6)


def extract_aligned_signals(traces_csv: Path, stimsnframes: np.ndarray,
                             frames_before: int = FRAMES_BEFORE,
                             frames_after:  int = FRAMES_AFTER,
                             normalization: str = NORMALIZATION) -> dict:
    """
    Extrae ventanas alineadas al estimulo y normaliza.
    Solo excluye trials genuinamente fuera de la grabacion (no por F0 bajo).
    """
    df = pd.read_csv(traces_csv, decimal=',', sep=None, engine='python')
    if df.columns[0].lower() in ['unnamed: 0', 'index', 'time', 'frame']:
        df.drop(df.columns[0], axis=1, inplace=True)
    df = df.apply(pd.to_numeric, errors='coerce').fillna(0)

    n_frames_total = len(df)
    total          = frames_before + frames_after
    extracted_data = {}
    n_excluded     = 0

    for ai in range(stimsnframes.shape[0]):
        for ti in range(stimsnframes.shape[1]):
            angle = int(stimsnframes[ai, ti, 0])
            sf    = int(stimsnframes[ai, ti, 1])
            start = sf - frames_before
            end   = sf + frames_after

            if start < 0 or end > n_frames_total:
                n_excluded += 1
                continue

            F      = df.iloc[start:end].values.copy()
            signal = normalize_trial(F, frames_before, normalization)
            extracted_data.setdefault(angle, []).append(signal)

    if n_excluded > 0:
        print(f'  Trials excluidos por borde: {n_excluded}')

    for angle in extracted_data:
        stacked = np.stack(extracted_data[angle], axis=-1)   # (T, N, trials)
        extracted_data[angle] = np.transpose(stacked, (1, 0, 2))  # (N, T, trials)

    return extracted_data


# ══════════════════════════════════════════════════════════════════════════════
#  MÓDULO 5 — Neuronas responsivas
# ══════════════════════════════════════════════════════════════════════════════

def find_responsive_neurons(extracted_data: dict,
                             threshold_std: float = RESPONSIVE_STD) -> list:
    """
    Neurona responsiva si en AL MENOS UN angulo,
    su respuesta supera threshold_std * std(baseline) en >= 50% de trials.
    """
    n_neurons  = next(iter(extracted_data.values())).shape[0]
    responsive = []

    for n in range(n_neurons):
        is_resp = False
        for angle, data in extracted_data.items():
            if is_resp:
                break
            d = data[n]   # (T, trials)
            n_resp_trials = 0
            for t in range(d.shape[1]):
                trial     = d[:, t]
                threshold = np.std(trial[:FRAMES_BEFORE]) * threshold_std
                if np.mean(trial[RESP_START:RESP_END]) > threshold:
                    n_resp_trials += 1
            if n_resp_trials >= max(1, d.shape[1] / 2):
                is_resp = True
        if is_resp:
            responsive.append(n)

    return responsive


# ══════════════════════════════════════════════════════════════════════════════
#  MÓDULO 6 — Tuning curve y 4 métodos de OSI
# ══════════════════════════════════════════════════════════════════════════════

def compute_tuning_curve(extracted_data: dict, neuron_idx: int) -> tuple:
    """Respuesta media +- sem por angulo. Devuelve (angles, means, sems)."""
    records = []
    for angle, data in extracted_data.items():
        trial_means = np.mean(data[neuron_idx, RESP_START:RESP_END, :], axis=0)
        for v in trial_means:
            records.append({'angle': angle, 'resp': v})
    dfg = pd.DataFrame(records).groupby('angle')['resp'].agg(['mean', 'sem']).reset_index()
    dfg = dfg.sort_values('angle')
    return dfg['angle'].values, dfg['mean'].values, dfg['sem'].values


def osi_vectorial(angles_deg: np.ndarray, responses: np.ndarray) -> float:
    """OSI vectorial = |Sum(R_k * exp(2i*theta_k))| / Sum(R_k). Rango [0,1]."""
    R = np.clip(responses, 0, None)
    if R.sum() == 0:
        return 0.0
    ang = np.deg2rad(2 * np.array(angles_deg))
    vec = np.sum(R * np.exp(1j * ang))
    return float(np.abs(vec) / R.sum())


def circular_variance(angles_deg: np.ndarray, responses: np.ndarray) -> float:
    """CV = 1 - OSI_vectorial. 0=selectiva, 1=no selectiva."""
    return 1.0 - osi_vectorial(angles_deg, responses)


def osi_classic(angles_deg: np.ndarray, responses: np.ndarray) -> tuple:
    """OSI clasico = (R_pref - R_ort)/(R_pref + R_ort). Devuelve (OSI, pref, DSI)."""
    angles = np.array(angles_deg)
    R      = np.array(responses)
    idx_pref   = np.argmax(R)
    pref_ang   = angles[idx_pref]
    rp         = R[idx_pref]
    target_ort = (pref_ang + 90) % 360
    target_opp = (pref_ang + 180) % 360
    diffs_ort  = np.abs(((angles - target_ort + 180) % 360) - 180)
    diffs_opp  = np.abs(((angles - target_opp + 180) % 360) - 180)
    ro  = R[np.argmin(diffs_ort)]
    rop = R[np.argmin(diffs_opp)]
    osi = (rp - ro)  / (rp + ro  + 1e-10)
    dsi = (rp - rop) / (rp + rop + 1e-10)
    return float(osi), float(pref_ang), float(dsi)


def osi_gmm(angles_deg: np.ndarray, responses: np.ndarray) -> tuple:
    """OSI via GMM bimodal. Siempre encuentra pico aunque la curva sea plana."""
    shifted = responses - np.min(responses)
    mx      = np.max(shifted)
    scaled  = (shifted / mx * 19 + 1) if mx > 0 else np.ones_like(shifted)
    dist = []
    for i, angle in enumerate(angles_deg):
        count = max(1, min(int(np.round(scaled[i])), 20))
        dist.extend([int(angle)] * count)
    if len(dist) < 4:
        return 0.0, float(angles_deg[0]), 0.0
    gmm   = GaussianMixture(n_components=2, random_state=0)
    gmm.fit(np.array(dist).reshape(-1, 1))
    theta = np.linspace(0, 360, 360)
    pdf   = np.exp(gmm.score_samples(theta.reshape(-1, 1))) * len(dist) * 30
    tp    = int(np.argmax(pdf))
    rp    = pdf[tp]
    ro    = pdf[(tp + 90) % 360]
    osi   = float((rp - ro) / (rp + ro + 1e-10))
    conf  = float(np.abs(gmm.weights_[0] - gmm.weights_[1]))
    return osi, float(tp), conf


def _bimodal_gaussian(theta, b, c, theta_pref, sigma, d):
    return (b
            + c * np.exp(-((theta - theta_pref) % 360)**2 / (2 * sigma**2))
            + d * np.exp(-((theta - theta_pref + 180) % 360)**2 / (2 * sigma**2)))


def osi_bimodal_gaussian(angles_deg, responses, r2_thresh=R2_THRESH):
    """Ajuste gaussiana bimodal (Ortiz-Cruz). Devuelve (OSI, pref, DSI, R2, valid)."""
    ang  = np.array(angles_deg, dtype=float)
    R    = np.array(responses,  dtype=float)
    idx0 = np.argmax(R)
    p0   = [np.min(R), np.max(R) - np.min(R), ang[idx0], 30.0,
            (np.max(R) - np.min(R)) * 0.3]
    bounds = ([-np.inf, 0, ang.min() - 10, 5, 0],
              [ np.inf, np.inf, ang.max() + 10, 120, np.inf])
    try:
        popt, _ = optimize.curve_fit(_bimodal_gaussian, ang, R,
                                     p0=p0, bounds=bounds, maxfev=5000)
        b, c, theta_pref, sigma, d = popt
        R_fit  = _bimodal_gaussian(ang, *popt)
        ss_res = np.sum((R - R_fit)**2)
        ss_tot = np.sum((R - np.mean(R))**2)
        r2     = 1 - ss_res / (ss_tot + 1e-10)
        if r2 < r2_thresh:
            return 0.0, float(ang[idx0]), 0.0, float(r2), False
        rp  = b + c
        rop = b + d
        osi = (rp - b) / (rp + b + 1e-10)
        dsi = (rp - rop) / (rp + rop + 1e-10)
        return float(osi), float(theta_pref % 180), float(dsi), float(r2), True
    except (RuntimeError, ValueError):
        return 0.0, float(ang[idx0]), 0.0, 0.0, False


def compute_osi_all_neurons(extracted_data: dict,
                             responsive_neurons: list) -> pd.DataFrame:
    records = []
    for n in responsive_neurons:
        angles, means, sems = compute_tuning_curve(extracted_data, n)
        osi_v                 = osi_vectorial(angles, means)
        cv                    = circular_variance(angles, means)
        osi_c, pref_c, dsi_c  = osi_classic(angles, means)
        osi_g, pref_g, conf_g = osi_gmm(angles, means)
        osi_b, pref_b, dsi_b, r2_b, vb = osi_bimodal_gaussian(angles, means)
        records.append({
            'neuron': n,
            'OSI_vec': osi_v,    'circ_var': cv,
            'OSI_classic': osi_c, 'DSI_classic': dsi_c, 'pref_classic': pref_c,
            'OSI_gmm': osi_g,    'pref_gmm': pref_g,    'gmm_conf': conf_g,
            'OSI_bimod': osi_b,  'DSI_bimod': dsi_b,    'pref_bimod': pref_b,
            'R2_bimod': r2_b,    'bimod_valid': vb,
        })
    return pd.DataFrame(records)


# ══════════════════════════════════════════════════════════════════════════════
#  MÓDULO 7 — Dataset poblacional
# ══════════════════════════════════════════════════════════════════════════════

def build_population_dataset(animal_data: dict) -> dict:
    """
    Dataset poblacional por genotipo.
    FIX: recorta trials al minimo comun entre animales antes de concatenar.
    """
    pop_data  = {}
    genotypes = set(d['genotype'] for d in animal_data.values())

    for geno in genotypes:
        animals_geno = {aid: d for aid, d in animal_data.items()
                        if d['genotype'] == geno}

        # Solo animales con neuronas responsivas
        animals_with_resp = {aid: d for aid, d in animals_geno.items()
                             if d['responsive']}
        if not animals_with_resp:
            print(f'  {geno}: sin neuronas responsivas, saltando.')
            continue

        all_angle_sets = [set(d['extracted_data'].keys())
                          for d in animals_with_resp.values()]
        common_angles  = sorted(set.intersection(*all_angle_sets))

        # Encontrar el minimo de trials por angulo entre todos los animales
        # para poder concatenar arrays de igual forma
        min_trials_per_angle = {}
        for a in common_angles:
            min_t = min(d['extracted_data'][a].shape[2]
                        for d in animals_with_resp.values())
            min_trials_per_angle[a] = min_t
        print(f'  {geno}: trials minimos por angulo = '
              f'{dict(zip(common_angles, [min_trials_per_angle[a] for a in common_angles]))}')

        resp_matrices, resp_by_angle = [], {a: [] for a in common_angles}
        animal_labels, neuron_ids, osi_dfs = [], [], []

        for aid, data in animals_with_resp.items():
            resp = data['responsive']
            ext  = data['extracted_data']
            osi_d = data['osi_df'].copy()
            osi_d['animal_id'] = aid
            osi_dfs.append(osi_d)

            # Respuesta media por angulo (N_resp, n_angles) — promedia trials
            rm = np.stack(
                [np.mean(ext[a][resp, RESP_START:RESP_END, :], axis=(1, 2))
                 for a in common_angles], axis=1
            )
            resp_matrices.append(rm)

            # Recortar trials al minimo comun para poder concatenar
            for a in common_angles:
                n_t = min_trials_per_angle[a]
                resp_by_angle[a].append(
                    ext[a][np.ix_(resp,
                                  list(range(RESP_START, RESP_END)),
                                  list(range(n_t)))]
                )

            animal_labels.extend([aid] * len(resp))
            neuron_ids.extend(resp)

        if not resp_matrices:
            continue

        pop_data[geno] = {
            'response_matrix': np.concatenate(resp_matrices, axis=0),
            'resp_by_angle':   {a: np.concatenate(resp_by_angle[a], axis=0)
                                for a in common_angles},
            'angles':          common_angles,
            'animal_labels':   np.array(animal_labels),
            'neuron_ids':      np.array(neuron_ids),
            'osi_df':          pd.concat(osi_dfs, ignore_index=True),
        }

    return pop_data


print('Funciones cargadas.')


Funciones cargadas.


In [4]:
animals = discover_animals(DATA_ROOT)
print(f'Animales encontrados: {len(animals)}')
for a in animals:
    print(f"  [{a['genotype']:6}]  {a['group']}/{a['animal_id']}")

Animales encontrados: 12
  [BTBR  ]  BTBR/BTBR_R2_060725
  [BTBR  ]  BTBR/BTBR_R2_270923_hembra_pasiva_Isaac
  [BTBR  ]  BTBR/BTBRL1030825_Reg_12_12_25
  [BTBR  ]  BTBR/BTBRR1_280825_Reg_15_12_25
  [BTBR  ]  BTBR/BTBRSM_270923_hembra_isaac
  [C57   ]  C57/C57_SM_Isaac
  [C57   ]  C57/C57L10300325_pasiva
  [C57   ]  C57/C57L2_Isaac
  [C57   ]  C57/C57R1230825_Reg_15_12_25
  [C57   ]  C57/C57R20010525_pasiva
  [C57   ]  C57/C57R3230825_Reg_15_12_25
  [C57   ]  C57/Delta 4-22 wt


In [5]:
all_results = []
animal_data = {}
failed      = []

for animal in animals:
    aid    = animal['animal_id']
    folder = animal['folder']
    geno   = animal['genotype']
    sep    = '=' * 60
    print(f'\n{sep}')
    print(f'  Procesando: {aid}  [{geno}]')
    print(sep)

    try:
        scope_file  = find_file(folder, SCOPE_KEYWORD)
        stim_file   = find_file(folder, STIM_KEYWORD)
        traces_file = find_file(folder, TRACES_KEYWORD)
        print(f'  scope  : {scope_file.name}')
        print(f'  stim   : {stim_file.name}')
        print(f'  traces : {traces_file.name}')

        ts_strs, ts_ms_scope, millisec, sensores = parse_scope_txt(scope_file)
        tstamp_stlog, ts_ms_stim, estimulos      = parse_stim_txt(stim_file)

        flags, counts = np.unique(sensores, return_counts=True)
        print(f'  Flags scope : {dict(zip(flags.astype(int), counts))}')
        sf, sc = np.unique(estimulos, return_counts=True)
        print(f'  Angulos/flags stim: {dict(zip(sf.astype(int), sc))}')

        t_clean = get_clean_frame_times(millisec, sensores)
        print(f'  Frames limpios: {len(t_clean)}')

        stimsnframes, n_unmatched = align_stimuli_to_frames(
            ts_ms_scope, millisec, sensores, ts_ms_stim, estimulos)

        frames_sf = stimsnframes[:, :, 1].flatten()
        print(f'  stimsnframes: {stimsnframes.shape}')
        print(f'  Angulos detectados: {stimsnframes[:, 0, 0].tolist()}')
        print(f'  Frame range: {int(frames_sf.min())} - {int(frames_sf.max())} / {len(t_clean)}')
        if n_unmatched > 0:
            print(f'  Onsets sin frame cercano (>500ms): {n_unmatched}')

        extracted_data = extract_aligned_signals(
            traces_file, stimsnframes, normalization=NORMALIZATION)

        if not extracted_data:
            print(f'  ERROR: extracted_data vacio - todos los trials fuera de rango')
            failed.append(aid)
            continue

        sample_angle = next(iter(extracted_data))
        n_neurons    = extracted_data[sample_angle].shape[0]
        print(f'  Angulos con datos: {sorted(extracted_data.keys())}')
        print(f'  Shape por angulo: {extracted_data[sample_angle].shape} (neuronas, frames, trials)')

        responsive = find_responsive_neurons(extracted_data)
        print(f'  Responsivas: {len(responsive)} / {n_neurons} ({100*len(responsive)/n_neurons:.1f}%)')

        osi_df = compute_osi_all_neurons(extracted_data, responsive)
        osi_df['animal_id']    = aid
        osi_df['genotype']     = geno
        osi_df['n_total']      = n_neurons
        osi_df['n_responsive'] = len(responsive)

        if len(osi_df) > 0:
            print(f'  OSI vectorial medio: {osi_df.OSI_vec.mean():.3f}')
            print(f'  OSI clasico medio:   {osi_df.OSI_classic.mean():.3f}')
            print(f'  Bimod validas (R2>{R2_THRESH}): {osi_df.bimod_valid.sum()}')

        all_results.append(osi_df)
        animal_data[aid] = {
            'extracted_data': extracted_data,
            'responsive':     responsive,
            'osi_df':         osi_df,
            'genotype':       geno,
        }
        osi_df.to_csv(OUTPUT_DIR / f'{aid}_osi_dsi.csv', index=False)
        print(f'  Guardado: {aid}_osi_dsi.csv')

    except Exception:
        print(f'  ERROR en {aid}:')
        traceback.print_exc()
        failed.append(aid)

sep = '=' * 60
print(f'\n{sep}')
print(f'Exitosos: {len(all_results)} / {len(animals)}')
if failed:
    print(f'Fallidos: {failed}')



  Procesando: BTBR_R2_060725  [BTBR]
  scope  : BTBRmachor2060725_reg112539.txt
  stim   : BTBRmachor2060725_pas112548.txt
  traces : machoR2_060725_BTBR_pasiva_OryDir_CTraces.csv
  Flags scope : {np.int64(2): np.int64(998), np.int64(4): np.int64(309), np.int64(5): np.int64(329), np.int64(23): np.int64(36803)}
  Angulos/flags stim: {np.int64(0): np.int64(10), np.int64(11): np.int64(90), np.int64(45): np.int64(10), np.int64(90): np.int64(10), np.int64(135): np.int64(10), np.int64(180): np.int64(10), np.int64(225): np.int64(10), np.int64(270): np.int64(10), np.int64(315): np.int64(10)}
  Frames limpios: 6070
  stimsnframes: (8, 10, 2)
  Angulos detectados: [0, 45, 90, 135, 180, 225, 270, 315]
  Frame range: 60 - 5219 / 6070
  Angulos con datos: [0, 45, 90, 135, 180, 225, 270, 315]
  Shape por angulo: (187, 30, 10) (neuronas, frames, trials)
  Responsivas: 139 / 187 (74.3%)
  OSI vectorial medio: 0.753
  OSI clasico medio:   0.890
  Bimod validas (R2>0.7): 91
  Guardado: BTBR_R2_060725_o

## 5. Imports y parámetros de ensambles

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import pdist
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

try:
    import umap as umap_lib
    HAS_UMAP = True
    print("UMAP disponible.")
except ImportError:
    HAS_UMAP = False
    print("UMAP no instalado — pip install umap-learn")

# Parámetros
ENS_N_PCA        = 10    # PCs máximos para reducción
ENS_UMAP_NEIGHBORS = 15
ENS_UMAP_MIN_DIST  = 0.1
ENS_GMM_MAX_K    = 8
ENS_RANDOM_STATE = 42


UMAP no instalado — pip install umap-learn


## 6. Construir matriz Trials × Neuronas

In [7]:
def build_ensemble_matrix(animal_data: dict) -> dict:
    """
    Para cada animal, construye:
      X (N_trials, N_neuronas) — actividad media por trial en ventana de respuesta
      angle_labels (N_trials,) — ángulo de cada trial
    Usa directamente extracted_data del pipeline (ya normalizado y alineado).
    """
    out = {}
    for aid, data in animal_data.items():
        ext  = data['extracted_data']   # {angulo: (N_neuronas, T, n_trials)}
        geno = data['genotype']
        vecs, angle_labels = [], []
        for angle in sorted(ext.keys()):
            arr  = ext[angle]                              # (N, T, trials)
            resp = np.mean(arr[:, RESP_START:RESP_END, :], axis=1)  # (N, trials)
            vecs.append(resp.T)                            # (trials, N)
            angle_labels.extend([angle] * arr.shape[2])
        if not vecs:
            continue
        X = np.concatenate(vecs, axis=0)                  # (N_trials, N_neuronas)
        out[aid] = {'X': X, 'angle_labels': np.array(angle_labels), 'genotype': geno}
        print(f"  {aid} [{geno}]: {X.shape[0]} trials × {X.shape[1]} neuronas")
    return out

ensemble_data = build_ensemble_matrix(animal_data)


  BTBR_R2_060725 [BTBR]: 80 trials × 187 neuronas
  BTBR_R2_270923_hembra_pasiva_Isaac [BTBR]: 64 trials × 93 neuronas
  BTBRL1030825_Reg_12_12_25 [BTBR]: 80 trials × 92 neuronas
  BTBRR1_280825_Reg_15_12_25 [BTBR]: 80 trials × 127 neuronas
  BTBRSM_270923_hembra_isaac [BTBR]: 64 trials × 20 neuronas
  C57_SM_Isaac [C57]: 64 trials × 66 neuronas
  C57L10300325_pasiva [C57]: 64 trials × 78 neuronas
  C57L2_Isaac [C57]: 64 trials × 57 neuronas
  C57R1230825_Reg_15_12_25 [C57]: 80 trials × 140 neuronas
  C57R20010525_pasiva [C57]: 64 trials × 146 neuronas
  C57R3230825_Reg_15_12_25 [C57]: 80 trials × 168 neuronas
  Delta 4-22 wt [C57]: 64 trials × 257 neuronas


## 7. PCA + UMAP + GMM

In [8]:
# ── Reconstruir ensemble_data usando solo neuronas responsivas ────────────────
print("Reconstruyendo matrices con solo neuronas responsivas...")

ensemble_data_resp = {}

for aid, data in animal_data.items():
    ext  = data['extracted_data']
    resp = data['responsive']
    geno = data['genotype']

    if not resp:
        print(f"  {aid}: sin responsivas, saltando")
        continue

    angles_sorted = sorted(ext.keys())
    vecs, angle_labels = [], []

    for angle in angles_sorted:
        arr  = ext[angle][resp, :, :]          # solo responsivas (N_resp, T, trials)
        r    = np.mean(arr[:, RESP_START:RESP_END, :], axis=1)  # (N_resp, trials)
        vecs.append(r.T)                        # (trials, N_resp)
        angle_labels.extend([angle] * arr.shape[2])

    X = np.concatenate(vecs, axis=0)
    ensemble_data_resp[aid] = {
        'X':            X,
        'angle_labels': np.array(angle_labels),
        'genotype':     geno,
    }
    print(f"  {aid} [{geno}]: {X.shape[0]} trials x {X.shape[1]} neuronas responsivas")

Reconstruyendo matrices con solo neuronas responsivas...
  BTBR_R2_060725 [BTBR]: 80 trials x 139 neuronas responsivas
  BTBR_R2_270923_hembra_pasiva_Isaac [BTBR]: 64 trials x 60 neuronas responsivas
  BTBRL1030825_Reg_12_12_25 [BTBR]: 80 trials x 7 neuronas responsivas
  BTBRR1_280825_Reg_15_12_25 [BTBR]: 80 trials x 70 neuronas responsivas
  BTBRSM_270923_hembra_isaac [BTBR]: 64 trials x 10 neuronas responsivas
  C57_SM_Isaac [C57]: 64 trials x 35 neuronas responsivas
  C57L10300325_pasiva [C57]: 64 trials x 49 neuronas responsivas
  C57L2_Isaac [C57]: 64 trials x 40 neuronas responsivas
  C57R1230825_Reg_15_12_25 [C57]: 80 trials x 11 neuronas responsivas
  C57R20010525_pasiva [C57]: 64 trials x 130 neuronas responsivas
  C57R3230825_Reg_15_12_25 [C57]: 80 trials x 75 neuronas responsivas
  Delta 4-22 wt [C57]: 64 trials x 94 neuronas responsivas


In [9]:
def reduce_and_cluster(X, max_k=ENS_GMM_MAX_K):
    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X)

    n_comp     = min(ENS_N_PCA, X_sc.shape[0] - 1, X_sc.shape[1])
    pca        = PCA(n_components=n_comp, random_state=ENS_RANDOM_STATE)
    pca_coords = pca.fit_transform(X_sc)
    exp_var    = pca.explained_variance_ratio_

    # UMAP sobre PCs que explican 90% de varianza
    umap_coords = None
    if HAS_UMAP:
        n90    = max(2, int(np.searchsorted(np.cumsum(exp_var), 0.90)) + 1)
        n90    = min(n90, pca_coords.shape[1])
        neigh  = min(ENS_UMAP_NEIGHBORS, pca_coords.shape[0] - 1)
        if neigh >= 2:
            reducer     = umap_lib.UMAP(n_components=2, n_neighbors=neigh,
                                        min_dist=ENS_UMAP_MIN_DIST,
                                        random_state=ENS_RANDOM_STATE)
            umap_coords = reducer.fit_transform(pca_coords[:, :n90])

    # GMM con selección de k por BIC
    bics, gmms = [], []
    for k in range(1, ENS_GMM_MAX_K + 1):
        g = GaussianMixture(n_components=k, covariance_type='full',
                            n_init=5, random_state=ENS_RANDOM_STATE)
        g.fit(pca_coords[:, :2])
        bics.append(g.bic(pca_coords[:, :2]))
        gmms.append(g)
    best_i   = int(np.argmin(bics))
    best_gmm = gmms[best_i]
    gmm_lbl  = best_gmm.predict(pca_coords[:, :2])
    sil      = silhouette_score(pca_coords[:, :2], gmm_lbl) if best_i > 0 else 0.0

    # Cosine similarity entre trials
    norms   = np.linalg.norm(X, axis=1, keepdims=True)
    X_norm  = X / (norms + 1e-10)
    cos_mat = X_norm @ X_norm.T
    mean_cs = float(np.mean(cos_mat[np.triu_indices_from(cos_mat, k=1)]))

    # Dispersión
    dists    = pdist(pca_coords[:, :2])
    centroid = np.mean(pca_coords[:, :2], axis=0)
    spread   = float(np.std(np.linalg.norm(pca_coords[:, :2] - centroid, axis=1)))

    return {
        'pca': pca_coords, 'exp_var': exp_var,
        'umap': umap_coords,
        'gmm_labels': gmm_lbl, 'best_k': best_i + 1,
        'bics': np.array(bics), 'silhouette': sil,
        'cos_mat': cos_mat, 'mean_cos_sim': mean_cs,
        'mean_pairwise': float(np.mean(dists)),
        'centroid_spread': spread,
    }


for aid, edata in ensemble_data_resp.items():
    # Limitar k maximo segun n_trials
    max_k = max(2, int(np.sqrt(edata['X'].shape[0] / 2)))
    print(f"  {aid}... (max_k={max_k})", end=' ', flush=True)

    res = reduce_and_cluster(edata['X'], max_k=max_k)
    edata['res'] = res
    print(f"k={res['best_k']}  sil={res['silhouette']:.2f}  "
          f"cos_sim={res['mean_cos_sim']:.3f}  spread={res['centroid_spread']:.4f}")


  BTBR_R2_060725... (max_k=6) k=7  sil=0.64  cos_sim=0.028  spread=5.0987
  BTBR_R2_270923_hembra_pasiva_Isaac... (max_k=5) k=5  sil=0.15  cos_sim=0.021  spread=2.4434
  BTBRL1030825_Reg_12_12_25... (max_k=6) k=4  sil=0.28  cos_sim=0.129  spread=1.1429
  BTBRR1_280825_Reg_15_12_25... (max_k=6) k=8  sil=0.70  cos_sim=0.055  spread=4.1631
  BTBRSM_270923_hembra_isaac... (max_k=5) k=5  sil=0.62  cos_sim=0.126  spread=1.3453
  C57_SM_Isaac... (max_k=5) k=8  sil=0.44  cos_sim=0.077  spread=2.2898
  C57L10300325_pasiva... (max_k=5) k=8  sil=0.30  cos_sim=0.055  spread=2.6891
  C57L2_Isaac... (max_k=5) k=6  sil=0.37  cos_sim=0.059  spread=3.7693
  C57R1230825_Reg_15_12_25... (max_k=6) k=6  sil=0.70  cos_sim=0.053  spread=1.9959
  C57R20010525_pasiva... (max_k=5) k=7  sil=0.52  cos_sim=0.201  spread=5.3098
  C57R3230825_Reg_15_12_25... (max_k=6) k=8  sil=0.29  cos_sim=0.027  spread=2.9556
  Delta 4-22 wt... (max_k=5) k=5  sil=0.37  cos_sim=0.032  spread=2.9224


## 8. Figuras por animal

In [10]:
angle_cmap = plt.cm.get_cmap('hsv', 8)

for aid, edata in ensemble_data.items():
    res     = edata['res']
    geno    = edata['genotype']
    color   = PALETTE.get(geno, '#7f8c8d')
    angles  = edata['angle_labels']
    u_ang   = sorted(np.unique(angles))
    ang_idx = {a: i for i, a in enumerate(u_ang)}
    pca2    = res['pca'][:, :2]
    k       = res['best_k']
    cg      = plt.cm.get_cmap('Set2', k)
    has_umap = res['umap'] is not None

    ncols = 4 if has_umap else 3
    fig   = plt.figure(figsize=(ncols * 4, 8))
    gs    = GridSpec(2, ncols, figure=fig, hspace=0.45, wspace=0.35)
    fig.suptitle(f'{aid}  [{geno}]', fontsize=13, fontweight='bold')

    # PCA — por ángulo
    ax = fig.add_subplot(gs[0, 0])
    for ang in u_ang:
        idx = angles == ang
        ax.scatter(pca2[idx, 0], pca2[idx, 1], s=18, alpha=0.6,
                   color=angle_cmap(ang_idx[ang] / max(1, len(u_ang)-1)), label=f'{ang}°')
    ax.set_xlabel(f'PC1 ({res["exp_var"][0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({res["exp_var"][1]*100:.1f}%)')
    ax.set_title('PCA — por ángulo')
    ax.legend(fontsize=6, ncol=2)
    ax.spines[['top','right']].set_visible(False)

    # PCA — por ensamble GMM
    ax = fig.add_subplot(gs[0, 1])
    for ki in range(k):
        idx = res['gmm_labels'] == ki
        ax.scatter(pca2[idx, 0], pca2[idx, 1], s=18, alpha=0.65,
                   color=cg(ki), label=f'E{ki+1}')
    ax.set_xlabel(f'PC1 ({res["exp_var"][0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({res["exp_var"][1]*100:.1f}%)')
    ax.set_title(f'GMM k={k}  sil={res["silhouette"]:.2f}')
    ax.legend(fontsize=7)
    ax.spines[['top','right']].set_visible(False)

    # UMAP
    next_col = 2
    if has_umap:
        ax = fig.add_subplot(gs[0, 2])
        uc = res['umap']
        for ang in u_ang:
            idx = angles == ang
            ax.scatter(uc[idx, 0], uc[idx, 1], s=18, alpha=0.6,
                       color=angle_cmap(ang_idx[ang] / max(1, len(u_ang)-1)), label=f'{ang}°')
        ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
        ax.set_title('UMAP — por ángulo')
        ax.legend(fontsize=6, ncol=2)
        ax.spines[['top','right']].set_visible(False)
        next_col = 3

    # Scree + BIC
    ax  = fig.add_subplot(gs[0, next_col])
    ax2 = ax.twinx()
    ns  = min(12, len(res['exp_var']))
    ax.bar(range(1, ns+1), res['exp_var'][:ns]*100, color=color, alpha=0.5)
    ax.plot(range(1, ns+1), np.cumsum(res['exp_var'][:ns])*100, 'o--', color=color, ms=4, lw=1.5)
    ax.axhline(90, ls=':', color='gray', lw=1)
    ax.set_ylabel('% Varianza', color=color)
    ax2.plot(range(1, ENS_GMM_MAX_K+1), res['bics'], 's-', color='#2c3e50', lw=1.5, ms=4)
    ax2.axvline(k, color='#2c3e50', ls='--', lw=1.2)
    ax2.set_ylabel('BIC', color='#2c3e50')
    ax.set_title('Scree / BIC')
    ax.spines[['top']].set_visible(False)

    # Cosine similarity heatmap
    ax = fig.add_subplot(gs[1, 0])
    sort_idx  = np.argsort(angles)
    cs_sorted = res['cos_mat'][np.ix_(sort_idx, sort_idx)]
    im = ax.imshow(cs_sorted, cmap='viridis', vmin=0, vmax=1, aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(f'Cos. sim. trials\n(media={res["mean_cos_sim"]:.3f})')
    ax.set_xlabel('Trial (ord. por ángulo)')
    ax.set_ylabel('Trial (ord. por ángulo)')

    # Dispersión intra-ángulo
    ax = fig.add_subplot(gs[1, 1])
    for ang in u_ang:
        pts = pca2[angles == ang]
        if len(pts) > 1:
            ax.hist(pdist(pts), bins=12, alpha=0.5, density=True,
                    color=angle_cmap(ang_idx[ang] / max(1, len(u_ang)-1)), label=f'{ang}°')
    ax.set_xlabel('Dist. euclídea en PCA1-2')
    ax.set_ylabel('Densidad')
    ax.set_title('Dispersión intra-ángulo')
    ax.legend(fontsize=6, ncol=2)
    ax.spines[['top','right']].set_visible(False)

    # Composición de ángulos por ensamble
    ax = fig.add_subplot(gs[1, 2])
    comp = np.zeros((k, len(u_ang)))
    for ki in range(k):
        for j, ang in enumerate(u_ang):
            comp[ki, j] = np.sum((res['gmm_labels'] == ki) & (angles == ang))
    comp_norm = comp / (comp.sum(axis=1, keepdims=True) + 1e-10)
    bottom = np.zeros(k)
    for j, ang in enumerate(u_ang):
        ax.bar(range(k), comp_norm[:, j], bottom=bottom,
               color=angle_cmap(j / max(1, len(u_ang)-1)), label=f'{ang}°', alpha=0.85)
        bottom += comp_norm[:, j]
    ax.set_xticks(range(k))
    ax.set_xticklabels([f'E{ki+1}\n(n={int(comp[ki].sum())})' for ki in range(k)], fontsize=8)
    ax.set_ylabel('Fracción de trials')
    ax.set_title('Composición por ensamble')
    ax.legend(fontsize=6, ncol=2, bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.spines[['top','right']].set_visible(False)

    # Trayectoria temporal
    if ncols == 4:
        ax = fig.add_subplot(gs[1, 3])
    else:
        ax = fig.add_subplot(gs[1, next_col])
    sc = ax.scatter(pca2[:, 0], pca2[:, 1], c=np.arange(len(pca2)),
                    cmap='plasma', s=18, alpha=0.7)
    plt.colorbar(sc, ax=ax, label='Trial #')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.set_title('Trayectoria temporal')
    ax.spines[['top','right']].set_visible(False)

    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / f'{aid}_ensembles.pdf', bbox_inches='tight')
    plt.show()


KeyError: 'res'

## 9. Comparación BTBR vs C57

In [ ]:
rows = []
for aid, edata in ensemble_data.items():
    res = edata['res']
    rows.append({
        'animal_id':       aid,
        'genotype':        edata['genotype'],
        'n_trials':        edata['X'].shape[0],
        'n_neuronas':      edata['X'].shape[1],
        'mean_cos_sim':    res['mean_cos_sim'],
        'n_ensambles':     res['best_k'],
        'silhouette':      res['silhouette'],
        'mean_pairwise':   res['mean_pairwise'],
        'centroid_spread': res['centroid_spread'],
    })

metrics = pd.DataFrame(rows)
metrics.to_csv(OUTPUT_DIR / 'ensemble_metrics.csv', index=False)
display(metrics.round(4))

genotypes = sorted(metrics['genotype'].unique())
cols_plot  = ['mean_cos_sim', 'mean_pairwise', 'centroid_spread']
labs_plot  = ['Coherencia\n(cos sim)', 'Distancia media\nentre trials', 'Dispersión\n(centroid spread)']

if len(metrics) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(11, 4))
    for ax, col, lbl in zip(axes, cols_plot, labs_plot):
        groups = [metrics.loc[metrics['genotype'] == g, col].values for g in genotypes]
        bp = ax.boxplot(groups, labels=genotypes, patch_artist=True, widths=0.5)
        for patch, g in zip(bp['boxes'], genotypes):
            patch.set_facecolor(PALETTE.get(g, '#95a5a6'))
            patch.set_alpha(0.65)
        for i, grp in enumerate(groups):
            ax.scatter(np.ones(len(grp))*(i+1) + np.random.normal(0, 0.05, len(grp)),
                       grp, color='black', s=30, zorder=4, alpha=0.7)
        ax.set_title(lbl, fontsize=9)
        ax.spines[['top','right']].set_visible(False)
        if len(genotypes) == 2 and all(len(g) > 1 for g in groups):
            _, p = stats.mannwhitneyu(groups[0], groups[1], alternative='two-sided')
            ax.set_xlabel(f'p = {p:.3f}', fontsize=9)
    fig.suptitle('Dispersión de ensambles por genotipo', fontsize=12)
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / 'ensemble_dispersion_comparison.pdf', bbox_inches='tight')
    plt.show()
else:
    print("Agrega más animales para la comparación entre genotipos.")


In [ ]:
# Cambia estos IDs por los animales que quieras mostrar en A y B
# Elige uno con resultados claros de cada genotipo

print("Animales disponibles:")
for aid, edata in ensemble_data.items():
    res = edata['res']
    print(f"  {aid} [{edata['genotype']}]  "
          f"cos_sim={res['mean_cos_sim']:.3f}  "
          f"spread={res['centroid_spread']:.3f}  "
          f"k={res['best_k']}  "
          f"sil={res['silhouette']:.2f}")


In [ ]:
# Edita aqui los animales representativos
REP_BTBR = None   # ej: 'BTBRL1_03_08_25'
REP_C57  = None   # ej: 'C57L1_30_03_25'

# Si no los defines, se elige automaticamente el de mayor silhouette por genotipo
for geno, rep_var in [('BTBR', 'REP_BTBR'), ('C57', 'REP_C57')]:
    if eval(rep_var) is None:
        best = max(
            [(aid, edata['res']['silhouette'])
             for aid, edata in ensemble_data.items()
             if edata['genotype'] == geno],
            key=lambda x: x[1]
        )
        if geno == 'BTBR':
            REP_BTBR = best[0]
        else:
            REP_C57 = best[0]

print('Representativo BTBR:', REP_BTBR)
print('Representativo C57: ', REP_C57)

In [ ]:
from matplotlib.gridspec import GridSpec
from scipy import stats
from scipy.spatial.distance import pdist

palette    = {'BTBR': '#e74c3c', 'C57': '#3498db'}
angle_cmap = plt.cm.get_cmap('hsv')
OUTPUT_DIR = Path('ensemble_figures')
OUTPUT_DIR.mkdir(exist_ok=True)

fig = plt.figure(figsize=(14, 11))
gs  = GridSpec(3, 4, figure=fig, hspace=0.5, wspace=0.4,
               height_ratios=[1.2, 1.2, 1.0])
fig.suptitle('Representacion poblacional de ensambles — BTBR vs C57',
             fontsize=13, fontweight='bold')

# ── Paleta de angulos ────────────────────────────────────────────────────────
all_angles = sorted(np.unique(
    np.concatenate([edata['angle_labels'] for edata in ensemble_data.values()])
))
n_ang = len(all_angles)
ang_idx = {a: i for i, a in enumerate(all_angles)}


def scatter_pca(ax, aid, title_suffix=''):
    edata  = ensemble_data[aid]
    res    = edata['res']
    geno   = edata['genotype']
    angles = edata['angle_labels']
    pca2   = res['pca'][:, :2]
    u_ang  = sorted(np.unique(angles))

    for ang in u_ang:
        idx = angles == ang
        c   = angle_cmap(ang_idx[ang] / max(n_ang-1, 1))
        ax.scatter(pca2[idx, 0], pca2[idx, 1],
                   color=c, alpha=0.6, s=20)

    ev = res['exp_var']
    ax.set_xlabel('PC1 (' + str(round(ev[0]*100, 1)) + '%)', fontsize=8)
    ax.set_ylabel('PC2 (' + str(round(ev[1]*100, 1)) + '%)', fontsize=8)
    ax.set_title(aid + ' [' + geno + ']' + title_suffix, fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


def heatmap_cos(ax, aid):
    edata   = ensemble_data[aid]
    res     = edata['res']
    angles  = edata['angle_labels']
    cos_mat = res['cos_mat']
    geno    = edata['genotype']

    sort_idx  = np.argsort(angles)
    cs_sorted = cos_mat[np.ix_(sort_idx, sort_idx)]
    mean_cs   = res['mean_cos_sim']

    im = ax.imshow(cs_sorted, cmap='viridis', vmin=0, vmax=1, aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title('Cos sim  media=' + str(round(mean_cs, 3)) +
                 '  [' + geno + ']', fontsize=8)
    ax.set_xlabel('Trial (ord. angulo)', fontsize=8)
    ax.set_ylabel('Trial (ord. angulo)', fontsize=8)


# ── Panel A: PCA representativos ─────────────────────────────────────────────
ax_a1 = fig.add_subplot(gs[0, 0:2])
ax_a2 = fig.add_subplot(gs[0, 2:4])
scatter_pca(ax_a1, REP_BTBR, ' — representativo')
scatter_pca(ax_a2, REP_C57,  ' — representativo')

# Leyenda de angulos compartida
handles_ang = [
    plt.Line2D([0],[0], marker='o', color='w',
               markerfacecolor=angle_cmap(ang_idx[a]/max(n_ang-1,1)),
               markersize=7, label=str(a)+'deg')
    for a in all_angles
]
ax_a1.legend(handles=handles_ang, fontsize=6, ncol=4,
             loc='upper right', framealpha=0.7)

# ── Panel B: Heatmap cosine similarity ───────────────────────────────────────
ax_b1 = fig.add_subplot(gs[1, 0:2])
ax_b2 = fig.add_subplot(gs[1, 2:4])
heatmap_cos(ax_b1, REP_BTBR)
heatmap_cos(ax_b2, REP_C57)

# ── Recopilar métricas de todos los animales ─────────────────────────────────
metrics_rows = []
for aid, edata in ensemble_data.items():
    res = edata['res']
    metrics_rows.append({
        'animal_id':       aid,
        'genotype':        edata['genotype'],
        'mean_cos_sim':    res['mean_cos_sim'],
        'centroid_spread': res['centroid_spread'],
        'mean_pairwise':   res['mean_pairwise'],
        'n_ensambles':     res['best_k'],
        'silhouette':      res['silhouette'],
    })
metrics = pd.DataFrame(metrics_rows)
genotypes = ['BTBR', 'C57']


def boxplot_metric(ax, col, ylabel, title):
    groups = [metrics.loc[metrics['genotype']==g, col].values for g in genotypes]
    bp = ax.boxplot(groups, labels=genotypes, patch_artist=True,
                    widths=0.5, showfliers=False,
                    medianprops=dict(color='black', lw=2))
    for patch, g in zip(bp['boxes'], genotypes):
        patch.set_facecolor(palette[g]); patch.set_alpha(0.7)
    for xi, (grp, g) in enumerate(zip(groups, genotypes)):
        jitter = np.random.normal(0, 0.06, len(grp))
        ax.scatter(np.ones(len(grp))*(xi+1) + jitter, grp,
                   color=palette[g], s=35, zorder=4, alpha=0.8,
                   edgecolors='black', linewidths=0.5)
    if len(groups[0]) > 1 and len(groups[1]) > 1:
        _, p = stats.mannwhitneyu(groups[0], groups[1], alternative='two-sided')
        ax.set_xlabel('Mann-Whitney p=' + str(round(p, 3)), fontsize=8)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


# ── Panel C: mean_cos_sim ─────────────────────────────────────────────────────
ax_c = fig.add_subplot(gs[2, 0])
boxplot_metric(ax_c, 'mean_cos_sim',
               'Cosine similarity media', 'C — Coherencia poblacional')

# ── Panel D: centroid_spread ──────────────────────────────────────────────────
ax_d = fig.add_subplot(gs[2, 1])
boxplot_metric(ax_d, 'centroid_spread',
               'Centroid spread (PCA12)', 'D — Dispersion de ensambles')

# ── Panel E: n_ensambles ──────────────────────────────────────────────────────
ax_e = fig.add_subplot(gs[2, 2])
boxplot_metric(ax_e, 'n_ensambles',
               'k optimo (GMM)', 'E — N ensambles')

# ── Panel F: silhouette ───────────────────────────────────────────────────────
ax_f = fig.add_subplot(gs[2, 3])
boxplot_metric(ax_f, 'silhouette',
               'Silhouette score', 'F — Cohesion de ensambles')

plt.savefig(OUTPUT_DIR / 'ensemble_comparison_figure.svg',
            format='svg', bbox_inches='tight')
plt.show()
print('Guardado en:', OUTPUT_DIR / 'ensemble_comparison_figure.svg')


In [ ]:
print('='*55)
print('RESUMEN ESTADISTICO — ENSAMBLES')
print('='*55)
for col, lbl in [('mean_cos_sim',    'Coherencia (cos sim)'),
                  ('centroid_spread', 'Dispersion (spread)'),
                  ('mean_pairwise',   'Distancia media'),
                  ('n_ensambles',     'N ensambles (k)'),
                  ('silhouette',      'Silhouette')]:
    g1 = metrics.loc[metrics['genotype']=='BTBR', col].values
    g2 = metrics.loc[metrics['genotype']=='C57',  col].values
    if len(g1) == 0 or len(g2) == 0: continue
    _, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
    print(lbl + ':')
    print('  BTBR  mean=' + str(round(float(np.mean(g1)),4)) +
          '  SEM=' + str(round(float(np.std(g1)/np.sqrt(len(g1))),4)) +
          '  n=' + str(len(g1)))
    print('  C57   mean=' + str(round(float(np.mean(g2)),4)) +
          '  SEM=' + str(round(float(np.std(g2)/np.sqrt(len(g2))),4)) +
          '  n=' + str(len(g2)))
    print('  Mann-Whitney p =', round(p, 4))
    print()

metrics.to_csv(OUTPUT_DIR / 'ensemble_metrics_summary.csv', index=False)
print('CSV guardado en:', OUTPUT_DIR / 'ensemble_metrics_summary.csv')


In [ ]:
# ── Clasificador SVM — decodificacion poblacional por orientacion ─────────────
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

print("Corriendo clasificador SVM (5-fold cross-validation)...")
svm_rows = []

for aid, data in animal_data.items():
    ext  = data['extracted_data']
    resp = data['responsive']
    geno = data['genotype']

    if not resp:
        continue

    # Construir X (trials, neuronas_resp) e y (angulo)
    vecs, labels = [], []
    for angle in sorted(ext.keys()):
        arr = ext[angle][resp, :, :]             # (N_resp, T, trials)
        r   = np.mean(arr[:, RESP_START:RESP_END, :], axis=1)  # (N_resp, trials)
        vecs.append(r.T)                          # (trials, N_resp)
        labels.extend([angle] * arr.shape[2])

    X = np.concatenate(vecs, axis=0)             # (N_trials, N_resp)
    y = np.array(labels)

    # Pipeline: escalar + SVM con kernel RBF
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('svm',    SVC(kernel='rbf', C=0.8, decision_function_shape='ovr'))
    ])

    # 5-fold estratificado
    cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores  = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')

    # Chance level = 1 / n_angulos
    chance  = 1 / len(np.unique(y))

    svm_rows.append({
        'animal_id':  aid,
        'genotype':   geno,
        'accuracy':   float(np.mean(scores)),
        'sem':        float(np.std(scores) / np.sqrt(len(scores))),
        'chance':     chance,
        'n_trials':   X.shape[0],
        'n_neuronas': X.shape[1],
    })
    print(f"  {aid} [{geno}]: accuracy={np.mean(scores)*100:.1f}% "
          f"(chance={chance*100:.1f}%)")

svm_df = pd.DataFrame(svm_rows)
display(svm_df.round(3))

# ── Figura ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 5))

genotypes = ['BTBR', 'C57']
for xi, geno in enumerate(genotypes):
    sub    = svm_df[svm_df['genotype'] == geno]
    vals   = sub['accuracy'].values * 100
    mean   = np.mean(vals)
    sem    = np.std(vals) / np.sqrt(len(vals))
    chance = sub['chance'].values[0] * 100

    ax.bar(xi, mean, width=0.5, color=palette[geno],
           alpha=0.8, edgecolor='black', linewidth=0.8)
    ax.errorbar(xi, mean, yerr=sem, color='black', capsize=5, lw=1.5)
    jitter = np.random.normal(0, 0.05, len(vals))
    ax.scatter(np.ones(len(vals))*xi + jitter, vals,
               color='black', s=40, zorder=5, alpha=0.8)

# Linea de chance
chance_level = svm_df['chance'].values[0] * 100
ax.axhline(chance_level, color='gray', ls='--', lw=1.5,
           label='Chance (' + str(round(chance_level, 1)) + '%)')

if len(svm_df[svm_df['genotype']=='BTBR']) > 1 and \
   len(svm_df[svm_df['genotype']=='C57'])  > 1:
    _, p = stats.mannwhitneyu(
        svm_df.loc[svm_df['genotype']=='BTBR', 'accuracy'].values,
        svm_df.loc[svm_df['genotype']=='C57',  'accuracy'].values,
        alternative='two-sided'
    )
    ax.set_xlabel('Mann-Whitney p=' + str(round(p, 3)), fontsize=9)

ax.set_xticks([0, 1])
ax.set_xticklabels(GENOTYPES if 'GENOTYPES' in dir() else genotypes)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Decodificacion SVM — orientacion visual')
ax.set_ylim(0, 105)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'svm_decoding.svg', format='svg', bbox_inches='tight')
plt.show()

# ── Resumen ───────────────────────────────────────────────────────────────────
print('='*50)
for geno in genotypes:
    sub = svm_df[svm_df['genotype']==geno]
    print(geno + ':  accuracy=' + str(round(sub['accuracy'].mean()*100,1)) +
          '%  SEM=' + str(round(sub['accuracy'].sem()*100,1)) + '%  n=' + str(len(sub)))
print('Chance level:', str(round(chance_level,1)) + '%')